# DistilBERT on SST-5 (3 classes): fine-tuning and faster inference

This notebook adds a third model to the project and compares it with the existing two
on the **same held-out test split** (2,203 sentences):

1. Fine-tune `distilbert-base-uncased` with mixed-precision (fp16) training, using the same
   train/validation/test files, the same 5 → 3 label grouping and the same class-weighted loss
   as the LSTM notebook. Keep the epoch with the best **validation macro F1**.
2. Evaluate on the test split and compare with the TF-IDF + logistic regression model and the LSTM.
3. Measure inference speed and GPU memory **before** (fp32, one sentence at a time, padded to 128 tokens)
   and **after** (batching, dynamic padding, length-sorted batches, fp16), and check that the faster
   setup still predicts the same labels.

**Run it in Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (about 5–10 minutes).
The last cell prints every number and saves them to `/content/distilbert_results.json`.

## 1. Setup

In [ ]:
import copy
import json
import random
import time
from collections import Counter
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from nltk import Tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import transformers
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
device = torch.device('cuda')
gpu = torch.cuda.get_device_name(0)
print(f'GPU: {gpu} | torch {torch.__version__} | transformers {transformers.__version__}')

## 2. Load SST-5 exactly like the LSTM notebook

Same source files, same label grouping (very negative + negative → negative, neutral → neutral,
positive + very positive → positive) and the same expected sentence counts, so every model is
scored on identical data.

In [ ]:
root = Path('/content/sst5')
root.mkdir(exist_ok=True)
BASE = 'https://raw.githubusercontent.com/stanfordnlp/sentiment-treebank/master/fiveclass'
LABELS = ['negative', 'neutral', 'positive']
LABEL_MAP = {0: 0, 1: 0, 2: 1, 3: 2, 4: 2}
EXPECTED = {'train': 8527, 'dev': 1095, 'test': 2203}

splits = {}
for split in ('train', 'dev', 'test'):
    path = root / f'{split}.txt'
    urlretrieve(f'{BASE}/{split}.txt', path)
    trees = [Tree.fromstring(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    rows = [(' '.join(tree.leaves()), LABEL_MAP[int(tree.label())]) for tree in trees]
    assert len(rows) == EXPECTED[split], f'{split}: expected {EXPECTED[split]}, found {len(rows)}'
    splits[split] = rows
    print(split, len(rows), dict(Counter(LABELS[y] for _, y in rows)))

## 3. Tokenize

DistilBERT uses its own WordPiece tokenizer. Sentences are tokenized once here; each training
batch is then padded only to its longest sentence (**dynamic padding**) instead of a fixed length.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode_rows(rows):
    encoded = tokenizer([text for text, _ in rows], truncation=True, max_length=MAX_LEN)
    return [{'input_ids': ids, 'label': label} for ids, (_, label) in zip(encoded['input_ids'], rows)]

def collate(batch):
    padded = tokenizer.pad({'input_ids': [item['input_ids'] for item in batch]}, return_tensors='pt')
    padded['labels'] = torch.tensor([item['label'] for item in batch])
    return padded

data = {split: encode_rows(rows) for split, rows in splits.items()}
test_lengths = [len(item['input_ids']) for item in data['test']]
print(f'Test sentences: {np.mean(test_lengths):.1f} tokens on average, longest {max(test_lengths)}')

## 4. Fine-tune with mixed precision

* AdamW, learning rate 2e-5 with linear warm-up/decay, 4 epochs, batch size 32, gradient clipping at 1.0.
* Forward pass in fp16 (`torch.autocast`) with a `GradScaler` so small fp16 gradients don't underflow to zero.
* Class-weighted cross-entropy, the same weights as the LSTM, because neutral is the smallest class.
* After each epoch, score the validation set and keep the weights with the best macro F1.

In [ ]:
EPOCHS, BATCH_SIZE, LR = 4, 32, 2e-5

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(data['train'], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, generator=generator)
dev_loader = DataLoader(data['dev'], batch_size=128, collate_fn=collate)
test_loader = DataLoader(data['test'], batch_size=128, collate_fn=collate)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
counts = Counter(label for _, label in splits['train'])
class_weights = torch.tensor([len(splits['train']) / (3 * counts[i]) for i in range(3)], device=device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler('cuda')

@torch.inference_mode()
def predict(net, loader, use_fp16):
    net.eval()
    actual, predicted = [], []
    for batch in loader:
        with torch.autocast('cuda', dtype=torch.float16, enabled=use_fp16):
            logits = net(input_ids=batch['input_ids'].to(device),
                         attention_mask=batch['attention_mask'].to(device)).logits
        actual += batch['labels'].tolist()
        predicted += logits.argmax(dim=-1).cpu().tolist()
    return actual, predicted

history, best_f1 = [], -1.0
torch.cuda.reset_peak_memory_stats()
train_start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_start, total_loss = time.perf_counter(), 0.0
    for batch in train_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
        loss = loss_fn(logits.float(), batch['labels'])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    actual, predicted = predict(model, dev_loader, use_fp16=True)
    val_f1 = f1_score(actual, predicted, average='macro')
    history.append({'epoch': epoch, 'train_loss': round(total_loss / len(train_loader), 4),
                    'val_accuracy': round(accuracy_score(actual, predicted), 4),
                    'val_macro_f1': round(val_f1, 4),
                    'epoch_seconds': round(time.perf_counter() - epoch_start, 1)})
    print(history[-1])
    if val_f1 > best_f1:
        best_f1, best_epoch = val_f1, epoch
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

train_seconds = time.perf_counter() - train_start
train_peak_gb = torch.cuda.max_memory_allocated() / 1e9
model.load_state_dict(best_state)
print(f'Selected epoch {best_epoch} (validation macro F1 {best_f1:.3f}). '
      f'Training took {train_seconds:.0f} s, peak GPU memory {train_peak_gb:.2f} GB.')

## 5. Test-set results and comparison

The reported DistilBERT score uses full fp32 inference. The committed TF-IDF model is downloaded from
this repository and re-scored on the same 2,203 test sentences; the LSTM numbers come from the README
(same split), because its checkpoint is not committed.

In [ ]:
test_y, test_p = predict(model, test_loader, use_fp16=False)
distilbert = {'accuracy': accuracy_score(test_y, test_p), 'macro_f1': f1_score(test_y, test_p, average='macro')}
print(classification_report(test_y, test_p, target_names=LABELS, digits=3))
print('Confusion matrix (rows = true, columns = predicted):')
print(confusion_matrix(test_y, test_p))

comparison = {'LSTM (README, same split)': {'accuracy': 0.533, 'macro_f1': 0.488}}
try:
    import joblib
    urlretrieve('https://github.com/lightyagami2109/Movie-Review-Sentiment-Classifier/raw/main/backend/'
                'sst5_tfidf_logreg_v2.joblib', '/content/tfidf.joblib')
    tfidf_pred = joblib.load('/content/tfidf.joblib').predict([text for text, _ in splits['test']])
    comparison['TF-IDF + LR (re-scored here)'] = {'accuracy': accuracy_score(test_y, tfidf_pred),
                                                  'macro_f1': f1_score(test_y, tfidf_pred, average='macro')}
except Exception as error:
    print('Could not re-score the TF-IDF model, using README numbers:', error)
    comparison['TF-IDF + LR (README, same split)'] = {'accuracy': 0.645, 'macro_f1': 0.579}
comparison['DistilBERT (this notebook)'] = distilbert

for name, scores in comparison.items():
    print(f"{name:32s} accuracy {scores['accuracy']:.3f}   macro F1 {scores['macro_f1']:.3f}")

print('\nExample DistilBERT mistakes:')
mistakes = [(text, truth, pred) for (text, truth), pred in zip(splits['test'], test_p) if truth != pred]
for text, truth, pred in mistakes[:5]:
    print(f'Expected {LABELS[truth]:8s} | Predicted {LABELS[pred]:8s} | {text}')

## 6. Inference benchmark: before vs after

Each setup classifies all 2,203 test sentences end to end (tokenization included) and is timed with
`torch.cuda.synchronize()` so queued GPU work is counted. Every setup runs 3 times and the fastest run is kept.

| Setup | What changes |
|---|---|
| **Before** | fp32 weights, 1 sentence per call, padded to 128 tokens |
| Batching | 64 sentences per call, padded only to the longest in the batch |
| + fp16 autocast | matrix multiplies in half precision on the T4's tensor cores |
| **After** | fp16 weights (`model.half()`), batches of 64 sorted by length so little padding is computed |

Memory is reported as the model weights plus the extra GPU memory each run needed. Correctness is checked
by comparing every predicted label with the fp32 "before" run.

In [ ]:
test_texts = [text for text, _ in splits['test']]
by_length = sorted(range(len(test_texts)), key=lambda i: len(data['test'][i]['input_ids']))
half_model = copy.deepcopy(model).half().eval()

def weights_mb(net):
    return sum(p.numel() * p.element_size() for p in net.parameters()) / 2**20

@torch.inference_mode()
def run(net, batch_size, use_fp16, pad_to_max=False, sort_by_length=False):
    order = by_length if sort_by_length else list(range(len(test_texts)))
    predictions = [None] * len(test_texts)
    net.eval()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    memory_before = torch.cuda.memory_allocated()
    start = time.perf_counter()
    for first in range(0, len(order), batch_size):
        index = order[first:first + batch_size]
        encoded = tokenizer([test_texts[i] for i in index], truncation=True, max_length=MAX_LEN,
                            padding='max_length' if pad_to_max else 'longest', return_tensors='pt').to(device)
        with torch.autocast('cuda', dtype=torch.float16, enabled=use_fp16):
            labels = net(**encoded).logits.argmax(dim=-1).tolist()
        for i, label in zip(index, labels):
            predictions[i] = label
    torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    return {'sentences_per_second': len(test_texts) / seconds,
            'ms_per_sentence': 1000 * seconds / len(test_texts),
            'total_seconds': seconds,
            'weights_mb': weights_mb(net),
            'extra_memory_mb': (torch.cuda.max_memory_allocated() - memory_before) / 2**20,
            'macro_f1': f1_score(test_y, predictions, average='macro'),
            'predictions': predictions}

setups = {
    'Before: fp32, batch 1, padded to 128': (model, dict(batch_size=1, use_fp16=False, pad_to_max=True)),
    'fp32, batch 64, dynamic padding': (model, dict(batch_size=64, use_fp16=False)),
    'fp16 autocast, batch 64, dynamic padding': (model, dict(batch_size=64, use_fp16=True)),
    'After: fp16 weights, batch 64, length-sorted': (half_model, dict(batch_size=64, use_fp16=False, sort_by_length=True)),
}

for net in (model, half_model):  # warm up CUDA kernels before timing
    run(net, 64, use_fp16=False)
    run(net, 64, use_fp16=True)

benchmark = {}
for name, (net, settings) in setups.items():
    benchmark[name] = min((run(net, **settings) for _ in range(3)), key=lambda r: r['total_seconds'])

reference = np.array(benchmark['Before: fp32, batch 1, padded to 128']['predictions'])
before = benchmark['Before: fp32, batch 1, padded to 128']
for name, result in benchmark.items():
    result['same_label_as_fp32_pct'] = float((np.array(result['predictions']) == reference).mean() * 100)
    result['speedup_vs_before'] = result['sentences_per_second'] / before['sentences_per_second']
    print(f"{name:46s} {result['sentences_per_second']:7.0f} sent/s ({result['speedup_vs_before']:5.1f}x) | "
          f"{result['ms_per_sentence']:6.2f} ms/sent | weights {result['weights_mb']:4.0f} MB + "
          f"{result['extra_memory_mb']:5.0f} MB | macro F1 {result['macro_f1']:.3f} | "
          f"same label as fp32 {result['same_label_as_fp32_pct']:.1f}%")

## 7. Save every number

In [ ]:
summary = {
    'environment': {'gpu': gpu, 'torch': torch.__version__, 'transformers': transformers.__version__},
    'training': {'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'learning_rate': LR, 'selected_epoch': best_epoch,
                 'best_val_macro_f1': round(best_f1, 4), 'train_seconds': round(train_seconds, 1),
                 'train_peak_gpu_gb': round(train_peak_gb, 2), 'history': history},
    'test_comparison': {name: {key: round(value, 4) for key, value in scores.items()}
                        for name, scores in comparison.items()},
    'inference': {name: {key: round(value, 3) for key, value in result.items() if key != 'predictions'}
                  for name, result in benchmark.items()},
}
Path('/content/distilbert_results.json').write_text(json.dumps(summary, indent=2))
print('RESULTS_JSON_START')
print(json.dumps(summary))
print('RESULTS_JSON_END')